# exp_e2_mpnet_multi — Semantic Graph Builder v2

**Phase 1a, embedder = `paraphrase-multilingual-mpnet-base-v2`, LLM = `deepseek-v32/latest` (API).**

Запускается из `exps/FINAL_EXPS/phase1a_embedders/exp_e2_mpnet_multi/` — все пути разрешаются автоматически от `clustering_1/`.

Перед запуском: `export YANDEX_CLOUD_API_KEY=...`.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

import torch

In [ ]:
import os, sys, logging
from pathlib import Path

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')

# layout: clustering_1/exps/FINAL_EXPS/phase1a_embedders/exp_e2_mpnet_multi/
EXP_DIR   = Path().resolve()
REPO_ROOT = EXP_DIR.parents[3]                 # clustering_1/
LLM_V2    = REPO_ROOT / 'llm_v2'

# put repo root on sys.path so `import llm_v2` works
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
# guard: never expose llm_v2/ as flat path
if str(LLM_V2) in sys.path:
    sys.path.remove(str(LLM_V2))

from llm_v2.config_schema import load_config
config = load_config(EXP_DIR / 'config.yaml')

# expand ${YANDEX_CLOUD_API_KEY} etc. (no-op for local LLMs)
config.llm.api_key = os.path.expandvars(config.llm.api_key)
config.llm.base_url = os.path.expandvars(config.llm.base_url)
config.llm.folder = os.path.expandvars(config.llm.folder)

print('EXP_DIR  :', EXP_DIR)
print('REPO_ROOT:', REPO_ROOT)
print('LLM_V2   :', LLM_V2)
print()
print(config.model_dump_json(indent=2))

EXP_DIR  : /home/platoon/graph/semantic-graph/exps/FINAL_EXPS/phase1b_llms/exp_l11_api_gemma3_27b
REPO_ROOT: /home/platoon/graph/semantic-graph
LLM_V2   : /home/platoon/graph/semantic-graph/llm_v2

{
  "llm": {
    "provider": "api",
    "model_name": "gemma-3-27b-it/latest",
    "max_new_tokens": 500,
    "temperature": 0.3,
    "device": "cpu",
    "load_in_8bit": false,
    "api_key": "${YANDEX_CLOUD_API_KEY}",
    "base_url": "https://ai.api.cloud.yandex.net/v1",
    "folder": "b1gpiug3vgbpe1cb4e5c",
    "instructions": ""
  },
  "embedding": {
    "model_name": "intfloat/multilingual-e5-large",
    "device": "cuda"
  },
  "coreference": {
    "enabled": false,
    "prompt_file": "prompts/coreference_ru.txt",
    "context_sentences": 3,
    "window_sentences": 5
  },
  "extraction": {
    "prompt_file": "prompts/extraction_ru.txt",
    "chunk_size": 3,
    "overlap_size": 1
  },
  "normalization": {
    "enabled": true,
    "language": "ru"
  },
  "deduplication": {
    "enabled": 

In [ ]:
config.llm.api_key = ""

In [ ]:
from llm_v2.models.llm_client import LLMClient
from llm_v2.models.embedder import Embedder

llm = LLMClient(config.llm)
embedder = Embedder(config.embedding)
print(f'LLM loaded: {config.llm.model_name}')
print(f'Embedder loaded: {config.embedding.model_name} (dim={embedder.dim})')

/home/platoon/graph/graph_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-05-07 16:53:24,591 [INFO] Load pretrained SentenceTransformer: intfloat/multilingual-e5-large
2026-05-07 16:53:24,871 [INFO] HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-large/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-05-07 16:53:24,902 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/intfloat/multilingual-e5-large/3d7cfbdacd47fdda877c5cd8a79fbcc4f2a574f3/modules.json "HTTP/1.1 200 OK"
2026-05-07 16:53:25,055 [INFO] HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-large/resolve/main/config_sentence_transformers.json "HTTP/1.1 404 Not Found"
2026-05-07 16:53:25,230 [INFO] HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e

LLM loaded: gemma-3-27b-it/latest
Embedder loaded: intfloat/multilingual-e5-large (dim=1024)


In [ ]:
from llm_v2.utils.io import load_text

input_path = Path(config.paths.input_text)
if not input_path.is_absolute():
    input_path = (LLM_V2 / input_path).resolve()
text = load_text(input_path)
print(f'Input: {input_path}')
print(f'Length: {len(text)} chars')
print(text[:500])

Input: /home/platoon/graph/semantic-graph/benchmark/final_bench/formated_fragment2.md
Length: 15466 chars
# Линейная классификация

Теперь давайте поговорим про задачу классификации. Для начала будем говорить про бинарную классификацию на два класса. Обобщить эту задачу до задачи классификации на $K$ классов не составит большого труда.

Пусть теперь наши таргеты $y$ кодируют принадлежность к положительному или отрицательному классу, то есть принадлежность множеству $\{-1,1\}$, а $x$ — по-прежнему векторы из $\mathbb{R}^D$.

В этом параграфе договоримся именно так обозначать классы, хотя в жизни вам 


## [0] Preprocessing

In [ ]:
from llm_v2.stages.preprocessing import preprocess

sentences = preprocess(text, language=config.normalization.language)
for s in sentences:
    print(f'  [{s.id}] {s.text}')

  [0] # Линейная классификация

Теперь давайте поговорим про задачу классификации.
  [1] Для начала будем говорить про бинарную классификацию на два класса.
  [2] Обобщить эту задачу до задачи классификации на $K$ классов не составит большого труда.
  [3] Пусть теперь наши таргеты $y$ кодируют принадлежность к положительному или отрицательному классу, то есть принадлежность множеству $\{-1,1\}$, а $x$ — по-прежнему векторы из $\mathbb{R}^D$.
  [4] В этом параграфе договоримся именно так обозначать классы, хотя в жизни вам будут нередко встречаться и метки $\{0,1\}$.
  [5] Мы хотим обучить линейную модель так, чтобы плоскость, которую она задаёт, как можно лучше отделяла объекты одного класса от объектов другого.
  [6] **2.1.6**

В идеальной ситуации найдётся плоскость, которая разделит классы: положительный окажется с одной стороны от неё, а отрицательный — с другой.
  [7] Выборка, для которой это возможно, называется линейно разделимой.
  [8] Увы, в реальной жизни такое встречается кр

## [1] Coreference Resolution

In [ ]:
from llm_v2.stages.coreference import resolve_coreferences

resolved_text, sentences = resolve_coreferences(
    sentences, llm, config.coreference, base_dir=LLM_V2
)
print('Resolved text:')
print(resolved_text)
print(f'\nSentences after coref: {len(sentences)}')

Resolved text:
# Линейная классификация

Теперь давайте поговорим про задачу классификации. Для начала будем говорить про бинарную классификацию на два класса. Обобщить эту задачу до задачи классификации на $K$ классов не составит большого труда. Пусть теперь наши таргеты $y$ кодируют принадлежность к положительному или отрицательному классу, то есть принадлежность множеству $\{-1,1\}$, а $x$ — по-прежнему векторы из $\mathbb{R}^D$. В этом параграфе договоримся именно так обозначать классы, хотя в жизни вам будут нередко встречаться и метки $\{0,1\}$. Мы хотим обучить линейную модель так, чтобы плоскость, которую она задаёт, как можно лучше отделяла объекты одного класса от объектов другого. **2.1.6**

В идеальной ситуации найдётся плоскость, которая разделит классы: положительный окажется с одной стороны от неё, а отрицательный — с другой. Выборка, для которой это возможно, называется линейно разделимой. Увы, в реальной жизни такое встречается крайне редко. Как обучить линейную модель

## [1.5] Chunking

In [ ]:
from llm_v2.stages.chunking import build_chunks

chunks = build_chunks(sentences, config.extraction)
for c in chunks:
    print(f'  {c.id} (sents {c.sentence_ids}): {c.text[:80]}...')

  chunk_0 (sents [0, 1, 2]): # Линейная классификация

Теперь давайте поговорим про задачу классификации. Для...
  chunk_1 (sents [2, 3, 4]): Обобщить эту задачу до задачи классификации на $K$ классов не составит большого ...
  chunk_2 (sents [4, 5, 6]): В этом параграфе договоримся именно так обозначать классы, хотя в жизни вам буду...
  chunk_3 (sents [6, 7, 8]): **2.1.6**

В идеальной ситуации найдётся плоскость, которая разделит классы: пол...
  chunk_4 (sents [8, 9, 10]): Увы, в реальной жизни такое встречается крайне редко. Как обучить линейную модел...
  chunk_5 (sents [10, 11, 12]): $$

<details>
<summary>Почему бы не решать задачу классификации как задачу регре...
  chunk_6 (sents [12, 13, 14]): Во вторых, ошибкой будет считаться предсказание, например, $5$ вместо $1$, хотя ...
  chunk_7 (sents [14, 15, 16]): </details>

Сконструируем теперь функционал ошибки так, чтобы он вышеперечисленн...
  chunk_8 (sents [16, 17, 18]): $$

Домножим обе части на $y_i$ и немного упростим:

$

## [2] Triplet Extraction

In [ ]:
from llm_v2.stages.extraction import extract_triplets

raw_triplets = extract_triplets(chunks, llm, config.extraction, base_dir=LLM_V2)
print(f'Extracted {len(raw_triplets)} raw triplets:')
for t in raw_triplets:
    print(f'  {t.subject} | {t.relation} | {t.object}  [{t.chunk_id}]')

Extracting triplets:   2%|▏         | 1/56 [07:11<6:35:27, 431.41s/it]


KeyboardInterrupt: 

## [3] Normalization

In [ ]:
from llm_v2.stages.normalization import normalize_triplets

norm_triplets = normalize_triplets(raw_triplets, config.normalization)
print(f'Normalized {len(norm_triplets)} triplets:')
for t in norm_triplets:
    print(f'  {t.norm_subject} | {t.norm_relation} | {t.norm_object}')

2026-05-07 15:34:07,871 [INFO] Loading dictionaries from /home/platoon/graph/graph_env/lib/python3.12/site-packages/pymorphy3_dicts_ru/data
2026-05-07 15:34:07,900 [INFO] format: 2.4, revision: 417150, updated: 2022-01-08T22:09:24.565962


Normalized 388 triplets:
  линейный классификация | являться | задача классификация
  задача классификация | мочь быть | бинарный классификация
  бинарный классификация | классифицировать на | два класс
  задача классификация | мочь быть обобщить до | задача классификация на k класс
  задача классификация на k класс | иметь количество класс | K
  задача классификация | обобщаться до | k класс
  таргет y | кодировать | принадлежность класс
  принадлежность | являться принадлежность множество | {-1,1}
  x | являться | вектор
  вектор | принадлежать пространство | ℝ^D
  класс | обозначаться как | -1 и 1
  метка класс | встречаться как | {0,1}
  мы | договориться обозначать | класс
  метка {0,1} | встречаться в жизнь | мы
  мы | хотеть обучить | линейный модель
  линейный модель | задавать | плоскость
  плоскость | отделять | объект один класс
  плоскость | отделять | объект другой класс
  плоскость | разделить | класс
  положительный класс | оказаться с один сторона | от плоскость
  отриц

## [4] Deduplication

In [ ]:
from llm_v2.stages.deduplication import deduplicate_triplets

dedup_triplets = deduplicate_triplets(norm_triplets, embedder, config.deduplication)
print(f'After dedup: {len(norm_triplets)} -> {len(dedup_triplets)} triplets')
for t in dedup_triplets:
    print(f'  {t.norm_subject} | {t.norm_relation} | {t.norm_object}')

After dedup: 388 -> 322 triplets
  линейный классификация | являться | задача классификация
  задача классификация | мочь быть | бинарный классификация
  бинарный классификация | классифицировать на | два класс
  задача классификация | обобщаться до | k класс
  таргет y | кодировать | принадлежность класс
  принадлежность | являться принадлежность множество | {-1,1}
  x | являться | вектор
  вектор | принадлежать пространство | ℝ^D
  класс | обозначаться как | -1 и 1
  метка класс | встречаться как | {0,1}
  мы | договориться обозначать | класс
  метка {0,1} | встречаться в жизнь | мы
  мы | хотеть обучить | линейный модель
  линейный модель | задавать | плоскость
  плоскость | отделять | объект один класс
  плоскость | разделить | класс
  положительный класс | оказаться с один сторона | от плоскость
  отрицательный класс | оказаться с другой сторона | от плоскость
  выборка | называться | линейно разделимый
  ситуация | встречаться | в реальный жизнь
  ситуация | встречаться | крайне 

## [5] Graph Assembly (raw)

In [ ]:
from llm_v2.stages.graph_assembly import assemble_graph

raw_graph = assemble_graph(dedup_triplets, chunks, text, config)
print(f'Raw graph: {len(raw_graph.nodes)} nodes, {len(raw_graph.edges)} edges')
print('\nNodes:')
for n in raw_graph.nodes:
    print(f'  {n.id}: {n.label} ({len(n.mentions)} mentions)')
print('\nEdges:')
for e in raw_graph.edges:
    print(f'  {e.id}: {e.source} --[{e.label}]--> {e.target} (w={e.weight})')

Raw graph: 335 nodes, 322 edges

Nodes:
  n0: линейный классификация (1 mentions)
  n1: задача классификация (3 mentions)
  n2: бинарный классификация (2 mentions)
  n3: два класс (1 mentions)
  n4: k класс (1 mentions)
  n5: таргет y (1 mentions)
  n6: принадлежность класс (1 mentions)
  n7: принадлежность (1 mentions)
  n8: {-1,1} (1 mentions)
  n9: x (1 mentions)
  n10: вектор (2 mentions)
  n11: ℝ^D (1 mentions)
  n12: класс (9 mentions)
  n13: -1 и 1 (1 mentions)
  n14: метка класс (3 mentions)
  n15: {0,1} (1 mentions)
  n16: мы (30 mentions)
  n17: метка {0,1} (1 mentions)
  n18: линейный модель (5 mentions)
  n19: плоскость (4 mentions)
  n20: объект один класс (1 mentions)
  n21: положительный класс (1 mentions)
  n22: от плоскость (2 mentions)
  n23: отрицательный класс (1 mentions)
  n24: выборка (2 mentions)
  n25: линейно разделимый (2 mentions)
  n26: ситуация (3 mentions)
  n27: в реальный жизнь (1 mentions)
  n28: крайне редко (1 mentions)
  n29: ошибка (2 mentions)
  n

## [6] Clustering

In [ ]:
from llm_v2.stages.clustering import cluster_graph, cluster_graph_multi, cluster_graph_all_methods
from llm_v2.utils.io import load_prompt

naming_prompt_path = Path(config.clustering.cluster_naming_prompt)
if not naming_prompt_path.is_absolute():
    naming_prompt_path = (LLM_V2 / naming_prompt_path).resolve()
naming_prompt = load_prompt(naming_prompt_path) if naming_prompt_path.exists() else None

if config.clustering.multi_method:
    multi = cluster_graph_all_methods(
        raw_graph, embedder, config,
        llm=llm, prompt_template=naming_prompt,
    )
    print('Multi-method clustering:')
    for method_name, mr in multi.methods.items():
        print(f'  {method_name}: {len(mr.param_labels)} variants')
        for lbl in mr.param_labels:
            g = mr.graphs[lbl]
            print(f'    {lbl}: {len(g.nodes)} nodes, {len(g.edges)} edges')
    agg = multi.methods['agglomerative']
    mid_label = agg.param_labels[len(agg.param_labels) // 2]
    clustered = agg.graphs[mid_label]
elif config.clustering.is_multi_threshold:
    multi = cluster_graph_multi(
        raw_graph, embedder, config,
        llm=llm, prompt_template=naming_prompt,
    )
    agg = multi.methods['agglomerative']
    print(f'Multi-threshold: {len(agg.param_labels)} levels')
    for lbl in agg.param_labels:
        g = agg.graphs[lbl]
        print(f'  t={lbl}: {len(g.nodes)} nodes, {len(g.edges)} edges')
    mid_label = agg.param_labels[len(agg.param_labels) // 2]
    clustered = agg.graphs[mid_label]
else:
    clustered = cluster_graph(
        raw_graph, embedder, config,
        llm=llm, prompt_template=naming_prompt,
    )

print(f'\nClustered graph: {len(clustered.nodes)} nodes, {len(clustered.edges)} edges')
print('\nClustered Nodes:')
for n in clustered.nodes:
    print(f'  {n.id}: {n.label} (members={n.members}, size={n.size})')
print('\nClustered Edges:')
for e in clustered.edges:
    print(f'  {e.id}: {e.source} --[{e.label}]--> {e.target} (size={e.size})')

## Save outputs

In [ ]:
from llm_v2.utils.io import save_json, save_text

out = EXP_DIR / config.paths.output_dir
out.mkdir(parents=True, exist_ok=True)

save_text(resolved_text, out / 'coreference_resolved.txt')
save_json(raw_graph.model_dump(), out / 'raw_graph.json')
save_json(clustered.model_dump(), out / 'clustered_graph.json')

if config.clustering.multi_method or config.clustering.is_multi_threshold:
    save_json(multi.model_dump(), out / 'multi_clustered_graph.json')
    method_counts = {m: len(r.param_labels) for m, r in multi.methods.items()}
    print(f'Saved multi_clustered_graph.json (methods: {method_counts})')

print(f'Saved to {out}/')

## Benchmark vs ground-truth graph

In [ ]:
from llm_v2.benchmark import (
    evaluate_graph,
    evaluate_multi_graph,
    load_clustered_graph,
    print_metrics,
    print_multi_metrics,
    best_variant,
    show_node_alignments,
    show_edge_alignments,
    multi_metrics_to_dict,
)

# GT inputs (absolute, robust to CWD)
gt_graph_path = REPO_ROOT / 'benchmark' / 'final_bench' / 'graph_clustered.json'
gt_text_path  = REPO_ROOT / 'benchmark' / 'final_bench' / 'formated_fragment2.md'

gt_graph = load_clustered_graph(gt_graph_path)
gt_text  = gt_text_path.read_text(encoding='utf-8')

# embedding context: prefer the coreference-resolved text the pipeline saw
source_text = resolved_text if resolved_text else gt_text

print(f'GT  : {len(gt_graph.nodes)} nodes, {len(gt_graph.edges)} edges  ({gt_graph_path})')
print(f'Pred: {len(clustered.nodes)} nodes, {len(clustered.edges)} edges')
print(f'Context text: {len(source_text)} chars')

TAU_NODE = 0.6
TAU_EDGE = 0.6
BETA = 1.0
NODE_WEIGHT = 0.6
EDGE_WEIGHT = 0.4
NODE_WINDOW = 300
EDGE_WINDOW = 400
TOP_K = 10

In [ ]:
metrics = evaluate_graph(
    pred=clustered,
    gt=gt_graph,
    source_text=source_text,
    embedder=embedder,
    tau_node=TAU_NODE,
    tau_edge=TAU_EDGE,
    beta=BETA,
    node_weight=NODE_WEIGHT,
    edge_weight=EDGE_WEIGHT,
    node_window=NODE_WINDOW,
    edge_window=EDGE_WINDOW,
)

print_metrics(metrics)
metrics.summary()

In [ ]:
show_node_alignments(clustered, gt_graph, metrics, top_k=TOP_K)

In [ ]:
show_edge_alignments(clustered, gt_graph, metrics, top_k=TOP_K)

In [ ]:
save_json(metrics.summary(), out / 'benchmark_metrics.json')
print(f'Saved benchmark_metrics.json to {out}/')

## Benchmark — multi-method / multi-threshold sweep

In [ ]:
is_multi = config.clustering.multi_method or config.clustering.is_multi_threshold

if not is_multi:
    print('Skipped: multi-method / multi-threshold not enabled in config')
    multi_metrics = None
else:
    total = sum(len(mr.graphs) for mr in multi.methods.values())
    print(f'Evaluating {total} configurations...')
    multi_metrics = evaluate_multi_graph(
        multi=multi,
        gt=gt_graph,
        source_text=source_text,
        embedder=embedder,
        tau_node=TAU_NODE,
        tau_edge=TAU_EDGE,
        beta=BETA,
        node_weight=NODE_WEIGHT,
        edge_weight=EDGE_WEIGHT,
        node_window=NODE_WINDOW,
        edge_window=EDGE_WINDOW,
    )
    print(f'Done: {sum(len(v) for v in multi_metrics.values())} variants evaluated')

In [ ]:
if multi_metrics:
    print_multi_metrics(multi_metrics, sort_by='graph_score')

In [ ]:
if multi_metrics:
    method, param, best_m = best_variant(multi_metrics, by='graph_score')
    best_graph = multi.methods[method].graphs[param]
    print(f'Best variant: method={method}, param={param}')
    print(f'  graph: {len(best_graph.nodes)} nodes, {len(best_graph.edges)} edges')
    print()
    print_metrics(best_m)

In [ ]:
if multi_metrics:
    save_json(multi_metrics_to_dict(multi_metrics), out / 'benchmark_metrics_multi.json')
    print(f'Saved benchmark_metrics_multi.json to {out}/')